# Procesamiento de audio con transformadas de fourier

Esta notebook muestra cómo aplicar un filtro pasa bajas a una señal de audio usando transformadas discretas de Fourier (DFT) en Python.

Puedes usarla para cargar un WAV, examinar su comportamiento en el dominio del tiempo y la frecuencia, filtrar las componentes no deseadas y guardar/reproducir el resultado desde el notebook.

Aquí verás paso a paso qué hace el proceso de filtrado:

- Leer un archivo WAV desde disco con `scipy.io.wavfile.read`.
- Convertir la señal a punto flotante normalizado para trabajar con la DFT.
- Calcular la transformada discreta de Fourier (DFT) usando `fft(data)`.
- Determinar el eje de frecuencias con `fftfreq(N, 1 / fs)`.
- Aplicar un filtro pasa bajas en el dominio de la frecuencia.
- Reconstruir la señal en el dominio del tiempo con `ifft(X_filtered)`.
- Guardar el resultado filtrado en un archivo WAV.
- Reproducir audio original y filtrado desde el notebook.

## ¿Qué hace exactamente la DFT?

La DFT convierte una señal de audio del dominio del tiempo al dominio de la frecuencia. Esto significa que en lugar de ver cómo cambia la amplitud con el tiempo, se ve cuánta energía hay en cada componente de frecuencia.

- `data`: es la señal original en el dominio del tiempo.
- `fs`: es la frecuencia de muestreo en hercios (Hz).
- `N`: es el número de muestras de la señal.
- `X = fft(data)`: es el espectro complejo de la señal.
- `freqs = fftfreq(N, 1 / fs)`: son las frecuencias asociadas a cada bin del espectro.
- `np.abs(X)`: es la magnitud de cada componente de frecuencia.
- `np.angle(X)`: es la fase de cada componente de frecuencia.
- `filter_mask = np.abs(freqs) <= cutoff_freq`: selecciona solo las componentes de frecuencia bajas.
- `X_filtered = X * filter_mask`: atenúa o elimina las frecuencias altas.
- `data_filtered = ifft(X_filtered)`: vuelve a transformar el espectro filtrado al dominio del tiempo.

### Por qué la DFT es útil aquí

- La DFT descompone la señal en un conjunto de senos y cosenos de diferentes frecuencias.
- Cada bin del espectro representa la contribución de una frecuencia específica a la señal original.
- Filtrar en el dominio de la frecuencia permite eliminar directamente las componentes no deseadas.
- La reconstrucción inversa con `ifft` devuelve una señal de audio adecuada para escucha.

## Detalles importantes de la DFT

- La DFT asume que la señal es periódica y se procesa en bloques finitos de tamaño `N`.
- `fftfreq(N, 1 / fs)` devuelve frecuencias positivas y negativas, porque la DFT es simétrica para señales reales.
- El filtro se aplica a ambas mitades del espectro para mantener la simetría y evitar distorsión al volver al dominio del tiempo.
- Al usar `np.real(ifft(...))`, se descarta una posible pequeña componente imaginaria que se genera por errores numéricos.

## Qué representa cada paso

1. `data = data.astype(np.float32)` y normalizar: lleva la señal al rango [-1, 1].
2. `fft(data)`: obtiene la representación en frecuencia.
3. `fftfreq(N, 1 / fs)`: calcula las frecuencias asociadas a cada elemento del espectro.
4. `filter_mask`: define qué frecuencias pasar y cuáles bloquear.
5. `X_filtered = X * filter_mask`: aplica el filtro multiplicando el espectro.
6. `ifft(X_filtered)`: transforma el espectro filtrado al dominio del tiempo.
7. `np.real(...)`: toma la parte real del resultado complejo.
8. Normalizar de nuevo y convertir a `int16` para guardar WAV.

## Qué verás en el notebook

- La señal original en el dominio del tiempo.
- La señal filtrada en el dominio del tiempo.
- La explicación de cómo se construye y aplica el filtro en frecuencia.
- La relación entre frecuencia de corte y contenido de audio.

## Cómo utilizar esta notebook

1. Coloca tu archivo WAV en la carpeta `uploads/`.
2. Cambia `input_path = 'uploads/tu_audio.wav'` por el nombre de tu archivo.
3. Ejecuta las celdas de código.
4. El audio filtrado se guarda en `processed/filtered_notebook.wav`.

Este notebook está pensado para aprender y experimentar con Fourier y filtrado de audio.


In [3]:
import numpy as np
import scipy.io.wavfile as wavfile
from scipy.fft import fft, ifft, fftfreq
from pathlib import Path
from IPython.display import Audio, display

def lowpass_filter(data, fs, cutoff_freq=4000):
    if data.ndim > 1:
        data = data[:, 0]  # usar canal izquierdo si es estéreo

    data = data.astype(np.float32)
    if data.dtype == np.int16:
        data = data / 32768.0

    N = len(data)
    X = fft(data)
    freqs = fftfreq(N, 1 / fs)

    filter_mask = np.abs(freqs) <= cutoff_freq
    X_filtered = X * filter_mask

    data_filtered = np.real(ifft(X_filtered))
    max_val = np.max(np.abs(data_filtered))
    if max_val > 0:
        data_filtered = data_filtered / max_val * 0.99

    data_output = (data_filtered * 32767).astype(np.int16)
    return data_output, data_filtered

def process_wav_file(input_path, output_path=None, cutoff_freq=4000):
    input_path = Path(input_path)
    if output_path is None:
        output_path = Path('processed') / f'notebook_filtered_{input_path.stem}.wav'
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fs, data = wavfile.read(str(input_path))
    data_output, data_filtered = lowpass_filter(data, fs, cutoff_freq)
    wavfile.write(str(output_path), fs, data_output)

    original_energy = float(np.sum(data.astype(np.float64) ** 2))
    filtered_energy = float(np.sum(data_filtered ** 2))

    return {
        'input': str(input_path),
        'output': str(output_path),
        'fs': int(fs),
        'duration_s': round(len(data_output) / fs, 2),
        'original_energy': original_energy,
        'filtered_energy': filtered_energy,
        'cutoff_freq': cutoff_freq
    }

In [4]:
# Ejemplo de uso: reemplaza 'uploads/tu_audio.wav' por el nombre de tu archivo WAV
input_path = 'processed/856648__skylosaurus__robot-bear-growl_1.wav'
output_path = 'uploads/tu_audio.wav'

result = process_wav_file(input_path, output_path, cutoff_freq=4000)
print(result)
display(Audio(result['output'], rate=result['fs']))

{'input': 'processed\\856648__skylosaurus__robot-bear-growl_1.wav', 'output': 'uploads\\tu_audio.wav', 'fs': 48000, 'duration_s': 7.1, 'original_energy': 3608516413784.0, 'filtered_energy': 5154.47998046875, 'cutoff_freq': 4000}


## Notas detalladas

- El archivo original debe existir en `uploads/` y debe ser un WAV legible por `scipy.io.wavfile`.
- El resultado filtrado se escribe en `processed/filtered_notebook.wav` por defecto.
- Si el audio es estéreo, el código toma solo el primer canal para simplificar el procesamiento.
- El filtro pasa bajas se aplica en el dominio de la frecuencia, lo cual es diferente de un filtro en el dominio del tiempo.
- La normalización antes y después de la DFT evita que el audio salga saturado tras la reconstrucción.
- En esta notebook se utiliza un valor de corte de `4000` Hz, pero puedes cambiarlo para experimentar con más o menos atenuación de agudos.

### Recomendaciones de uso

- Usa archivos WAV cortos o medianos para que el proceso sea rápido.
- Si quieres ver ondas o espectros, puedes agregar celdas propias con `matplotlib` y `np.abs(fft(...))`.
- La versión actual está pensada como una guía didáctica: lee las secciones de texto antes de ejecutar cada celda.
